In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.models import resnext50_32x4d
import torchvision.transforms as transforms
import time
import numpy as np
from tqdm import tqdm
from torchmetrics.functional import calibration_error

In [2]:
# =========================================================
# DEVICE SETUP
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("✅ Using MPS device.")
else:
    device = torch.device("cpu")
    print("⚠️ MPS device not found. Using CPU.")

✅ Using MPS device.


In [3]:
# =========================================================
# DATASET
# =========================================================
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=100, shuffle=False, num_workers=2)

Files already downloaded and verified
Files already downloaded and verified


In [4]:
# =========================================================
# MODEL SETUP (ResNeXt-50)
# =========================================================

# Initialize model for 10 classes
model = resnext50_32x4d(weights=None, num_classes=10).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [5]:
# =========================================================
# METRIC FUNCTIONS
# =========================================================
def topk_accuracy(output, target, topk=(1,)):
    maxk = max(topk)
    _, pred = output.topk(maxk, 1, True, True)
    correct = pred.eq(target.view(-1, 1).expand_as(pred))
    res = []
    for k in topk:
        correct_k = correct[:, :k].sum().item()
        res.append(correct_k)
    return res

def mean_off_diagonal_covariance(features, eps=1e-8):
    """Mean Off-diagonal Covariance (MOC) ↓"""
    X = features - features.mean(dim=0, keepdim=True)
    cov = (X.T @ X) / (X.shape[0] - 1)
    var = cov.diag()
    denom = torch.sqrt(var.unsqueeze(0) * var.unsqueeze(1)).clamp_min(eps)
    corr = cov / denom
    corr.fill_diagonal_(0)
    return corr.abs().mean().item()

def linear_CKA(X, Y):
    """Centered Kernel Alignment between two feature matrices"""
    X = X - X.mean(0, keepdim=True)
    Y = Y - Y.mean(0, keepdim=True)
    hsic = torch.norm(X.T @ Y, p='fro') ** 2
    var1 = torch.norm(X.T @ X, p='fro') * torch.norm(Y.T @ Y, p='fro')
    return (hsic / var1).item()

In [6]:
# =========================================================
# TRAINING
# =========================================================
num_epochs = 100
best_acc = 0
best_val_loss = float('inf')
convergence_epoch = None
patience = 5
no_improve_epochs = 0
train_losses, test_losses = [], []
start_time = time.time()

for epoch in range(num_epochs):
    model.train()
    running_loss, total, correct_top1, correct_top5 = 0.0, 0, 0, 0
    epoch_start = time.time()

    for inputs, labels in tqdm(trainloader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        top1, top5 = topk_accuracy(outputs, labels, topk=(1, 5))
        correct_top1 += top1
        correct_top5 += top5
        total += labels.size(0)

    train_loss = running_loss / len(trainloader)
    train_acc1 = 100 * correct_top1 / total
    train_acc5 = 100 * correct_top5 / total
    train_losses.append(train_loss)

    # =====================================================
    # VALIDATION
    # =====================================================
    model.eval()
    correct_top1, correct_top5, total = 0, 0, 0
    test_loss = 0.0
    probs, targets, features = [], [], []

    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            test_loss += loss.item()

            top1, top5 = topk_accuracy(outputs, labels, topk=(1, 5))
            correct_top1 += top1
            correct_top5 += top5
            total += labels.size(0)

            probs.append(F.softmax(outputs, dim=1).cpu())
            targets.append(labels.cpu())

            # Extract features before classifier
            feats = model.layer4(model.layer3(model.layer2(model.layer1(model.relu(model.bn1(model.conv1(inputs)))))))
            feats = torch.flatten(feats, 1).cpu()
            features.append(feats)

    test_loss /= len(testloader)
    test_losses.append(test_loss)
    test_acc1 = 100 * correct_top1 / total
    test_acc5 = 100 * correct_top5 / total
    train_test_gap = abs(train_acc1 - test_acc1)

    # --- Metrics ---
    all_probs = torch.cat(probs)
    all_targets = torch.cat(targets)
    ece = calibration_error(all_probs, all_targets, n_bins=15, task="multiclass", num_classes=10).item()

    all_features = torch.cat(features, dim=0)
    moc = mean_off_diagonal_covariance(all_features)

    epoch_time = time.time() - epoch_start

    print(f"\n📊 Epoch {epoch+1}:")
    print(f"Train Loss: {train_loss:.4f} | Train Acc@1: {train_acc1:.2f}% | Train Acc@5: {train_acc5:.2f}%")
    print(f"Val Loss: {test_loss:.4f} | Val Acc@1: {test_acc1:.2f}% | Val Acc@5: {test_acc5:.2f}%")
    print(f"Train–Test Gap: {train_test_gap:.2f}% | ECE: {ece:.4f} | MOC: {moc:.4f}")
    print(f"⏱ Epoch Time: {epoch_time:.2f}s")

    # Early stopping check
    if test_loss < best_val_loss - 1e-4:  
        best_val_loss = test_loss
        no_improve_epochs = 0
    else:
        no_improve_epochs += 1

    if no_improve_epochs >= patience:
        print(f"\n⏹ Early stopping triggered after {epoch+1} epochs (no improvement in {patience} epochs).")
        break

    # Save best model
    if test_acc1 > best_acc:
        best_acc = test_acc1
        convergence_epoch = epoch + 1
        torch.save(model.state_dict(), "best_resnext50_cifar10_mps.pth")

total_time = time.time() - start_time
print(f"\n✅ Training Complete.")
print(f"Best Top-1 Accuracy: {best_acc:.2f}% at Epoch {convergence_epoch}")
print(f"Total Training Time: {total_time/60:.2f} minutes")

# =========================================================
# INFERENCE TIME
# =========================================================
model.eval()
dummy_input = torch.randn(1, 3, 32, 32).to(device)
start = time.time()
_ = model(dummy_input)
inference_time = time.time() - start
print(f"\n⚡ Inference Time (1 sample): {inference_time*1000:.3f} ms")

Epoch 1/100: 100%|██████████| 391/391 [01:23<00:00,  4.70it/s]



📊 Epoch 1:
Train Loss: 1.9259 | Train Acc@1: 33.45% | Train Acc@5: 84.08%
Val Loss: 1.7765 | Val Acc@1: 41.44% | Val Acc@5: 91.26%
Train–Test Gap: 7.99% | ECE: 0.0732 | MOC: 0.4612
⏱ Epoch Time: 108.18s


Epoch 2/100: 100%|██████████| 391/391 [01:17<00:00,  5.03it/s]



📊 Epoch 2:
Train Loss: 1.6477 | Train Acc@1: 44.06% | Train Acc@5: 91.41%
Val Loss: 1.5023 | Val Acc@1: 46.45% | Val Acc@5: 92.50%
Train–Test Gap: 2.39% | ECE: 0.0434 | MOC: 0.5412
⏱ Epoch Time: 102.03s


Epoch 3/100: 100%|██████████| 391/391 [01:17<00:00,  5.04it/s]


...


📊 Epoch 44:
Train Loss: 0.3460 | Train Acc@1: 87.83% | Train Acc@5: 99.70%
Val Loss: 0.5657 | Val Acc@1: 83.03% | Val Acc@5: 99.07%
Train–Test Gap: 4.80% | ECE: 0.0536 | MOC: 0.1253
⏱ Epoch Time: 115.18s


Epoch 45/100: 100%|██████████| 391/391 [01:28<00:00,  4.40it/s]



📊 Epoch 45:
Train Loss: 0.3420 | Train Acc@1: 87.92% | Train Acc@5: 99.66%
Val Loss: 0.5136 | Val Acc@1: 83.48% | Val Acc@5: 99.07%
Train–Test Gap: 4.44% | ECE: 0.0443 | MOC: 0.1067
⏱ Epoch Time: 115.18s


Epoch 46/100: 100%|██████████| 391/391 [01:28<00:00,  4.41it/s]



📊 Epoch 46:
Train Loss: 0.3269 | Train Acc@1: 88.43% | Train Acc@5: 99.70%
Val Loss: 0.5150 | Val Acc@1: 83.74% | Val Acc@5: 99.02%
Train–Test Gap: 4.69% | ECE: 0.0504 | MOC: 0.1080
⏱ Epoch Time: 115.15s

⏹ Early stopping triggered after 46 epochs (no improvement in 5 epochs).

✅ Training Complete.
Best Top-1 Accuracy: 83.48% at Epoch 45
Total Training Time: 81.63 minutes

⚡ Inference Time (1 sample): 1260.604 ms
